# Portfolio Construction Setup

Load the working alpha functions from `Factors.ipynb`, load only required data inputs from `data/alpha_inputs`, and build one `alphas` dictionary.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from notebooks.operators_library import (
    ts_mean,
    ts_av_diff,
    ts_std_dev,
    ts_sum,
    ts_zscore,
    zscore,
    rank,
    ts_rank,
    signed_power,
    bucket,
    densify,
    ts_delay,
    group_neutralize,
    scale,
    trade_when,
    trade_when_hold,
    hump,
    vector_neut,
    group_mean,
    quantile_cauchy,
    ts_quantile_cauchy,
    ts_regression,
    ts_step,
    group_rank,
    winsorize,
    ts_backfill,
    ts_arg_min,
    ts_decay_linear,
    if_else,
    group_scale,
    days_from_last_change,
)

if (Path.cwd() / "notebooks").exists():
    repo_root = Path.cwd()
elif (Path.cwd().parent / "notebooks").exists():
    repo_root = Path.cwd().parent
else:
    raise FileNotFoundError("Could not locate repo root containing notebooks/.")


In [ ]:
WORKING_ALPHA_ORDER = [
    "alpha_0413a",
    "alpha_0615a",
    "alpha_0403c",
    "alpha_0812a",
    "alpha_0407a",
    "alpha_0612d",
    "alpha_0503b",
    "alpha_0412",
    "alpha_0418b",
    "alpha_0505",
    "alpha_0404b",
    "alpha_0618e",
    "alpha_0413b",
    "alpha_0810a",
    "alpha_0415b",
    "alpha_0617a",
    "alpha_0604c",
    "alpha_0616a",
]

factors_nb_path = repo_root / "notebooks" / "Factors.ipynb"
factors_nb = json.loads(factors_nb_path.read_text(encoding="utf-8"))

loaded_alpha_names = []
for cell in factors_nb.get("cells", []):
    if cell.get("cell_type") != "code":
        continue
    src = "".join(cell.get("source", []))
    stripped = src.strip()
    if not stripped.startswith("def alpha_"):
        continue
    alpha_name = stripped.split("def ")[1].split("(")[0].strip()
    if alpha_name not in WORKING_ALPHA_ORDER:
        continue

    exec(src, globals())
    loaded_alpha_names.append(alpha_name)

missing_alpha_defs = sorted(set(WORKING_ALPHA_ORDER) - set(loaded_alpha_names))
if missing_alpha_defs:
    raise ValueError(f"Missing alpha definitions in Factors.ipynb: {missing_alpha_defs}")

print(f"Loaded {len(loaded_alpha_names)} alpha functions from {factors_nb_path.name}.")


In [ ]:
ALPHA_INPUTS = {
    "alpha_0413a": ("close_df", "high_df", "low_df"),
    "alpha_0615a": ("volume_df", "sharesout_df", "cap_df"),
    "alpha_0403c": ("volume_df", "sharesout_df"),
    "alpha_0812a": ("liabilities_df", "assets_df", "debt_df", "equity_df"),
    "alpha_0407a": ("debt_df",),
    "alpha_0612d": ("debt_df", "assets_df", "volume_df"),
    "alpha_0503b": ("sharesout_df", "volume_df", "vwap_df", "low_df", "open_df", "high_df"),
    "alpha_0412": ("operating_income_df", "vwap_df", "volume_df", "returns_df"),
    "alpha_0418b": ("operating_income_df", "vwap_df"),
    "alpha_0505": ("operating_income_df", "vwap_df"),
    "alpha_0404b": ("returns_df", "operating_income_df", "cap_df"),
    "alpha_0618e": ("vwap_df", "close_df", "volume_df"),
    "alpha_0413b": ("operating_income_df", "cap_df"),
    "alpha_0810a": ("returns_df", "cap_df", "volume_df"),
    "alpha_0415b": ("close_df", "volume_df"),
    "alpha_0617a": ("open_df", "volume_df"),
    "alpha_0604c": ("close_df", "high_df", "low_df"),
    "alpha_0616a": ("close_df", "open_df", "volume_df"),
}

manifest_path = repo_root / "notebooks" / "data" / "alpha_inputs" / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest_files = manifest.get("files", {})

required_data_names = sorted({name for items in ALPHA_INPUTS.values() for name in items})
data = {}
for data_name in required_data_names:
    rel_path = manifest_files.get(data_name)
    if rel_path is None:
        rel_path = f"data/alpha_inputs/{data_name}.parquet"
    file_path = (repo_root / "notebooks" / rel_path).resolve()
    data[data_name] = pd.read_parquet(file_path)

globals().update(data)
print(f"Loaded {len(data)} input dataframes from {manifest_path.parent}.")


In [ ]:
alpha_functions = {name: globals()[name] for name in WORKING_ALPHA_ORDER}

alphas = {
    alpha_name: alpha_functions[alpha_name](
        *[data[input_name] for input_name in ALPHA_INPUTS[alpha_name]]
    )
    for alpha_name in WORKING_ALPHA_ORDER
}

print(f"Built alpha dictionary with {len(alphas)} alphas.")
print(list(alphas.keys()))
